# Coverage Plots
> Plot Nanopore read coverage from BAM files for genes and transcripts of interest

In [ ]:
#| default_exp coverage_plots

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import os
import warnings
import copy
import io
import logging
from datetime import datetime
from typing import Sequence, List, Optional, Mapping

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib import gridspec
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.backends.backend_pdf import FigureCanvasPdf

# Configure logging BEFORE importing trackplot
# Trackplot uses loguru, so we need to disable it at the module level
import sys

# Suppress loguru output by removing all handlers before trackplot import
try:
    from loguru import logger as loguru_logger
    loguru_logger.remove()  # Remove all handlers
    # Optionally re-add with higher level
    # loguru_logger.add(sys.stderr, level="ERROR")
except ImportError:
    pass

# Also configure standard logging (belt and suspenders)
logging.getLogger("trackplot").setLevel(logging.CRITICAL)
logging.getLogger("trackplot.plot").setLevel(logging.CRITICAL)
logging.getLogger("trackplot.file").setLevel(logging.CRITICAL)
logging.getLogger("trackplot.file.Annotation").setLevel(logging.CRITICAL)
logging.getLogger("__main__").setLevel(logging.CRITICAL)

from trackplot.plot import Plot
import trackplot.plot as _tp

## Compact Layout Monkey-Patch

This monkey-patch improves trackplot layouts by:
- Making the x-axis ruler thinner
- Reducing vertical spacing between tracks
- Pulling annotation tracks closer to data

In [ ]:
#| export
# Pull names from trackplot.plot
PlotInfo = _tp.PlotInfo
Depth = _tp.Depth
logger = _tp.logger

init_graph_coords = _tp.init_graph_coords
plot_density = _tp.plot_density
plot_hic = _tp.plot_hic
plot_site_plot = _tp.plot_site_plot
plot_heatmap = _tp.plot_heatmap
plot_line = _tp.plot_line
plot_igv_like = _tp.plot_igv_like
plot_motif = _tp.plot_motif
plot_links = _tp.plot_links
plot_annotation = _tp.plot_annotation
plot_stroke = _tp.plot_stroke
set_indicator_lines = _tp.set_indicator_lines
set_focus = _tp.set_focus
set_x_ticks = _tp.set_x_ticks



#| export
def _compact_plot(
    self,
    output: str | None = None,
    annotation_scale: int | float = 0.25,
    stroke_scale: int | float = 0.25,
    dpi: int = 300,
    width: int | float = 0,
    height: int | float = 0,
    raster: bool = False,
    return_image: str | None = None,
    sc_height_ratio: dict | None = None,
    distance_between_label_axis: float = 0.3,
    n_jobs: int = 1,
    fill_step: str = "post",
    grid_hspace: float = 0.02,
    *args,
    **kwargs,
):
    """Compact layout version of Plot.plot with extra layout controls."""
    if sc_height_ratio is None:
        sc_height_ratio = {"density": 0.2, "heatmap": 0.2}

    assert self.region is not None, "please set the plotting region first."

    plots_n_rows, plots_n_cols = 1, 1
    height_ratio: list[float] = []

    if self.annotation is not None:
        logger.info("load annotation")
        self.annotation.load(self.region, *args, **self.params["annotation"])
        plots_n_rows += self.annotation.len(scale=annotation_scale)

    if self.stroke:
        plots_n_rows += int(max(len(self.stroke) * stroke_scale, 1))

    if self.link:
        plots_n_rows += len(self.link)

    if self.sequence is not None:
        logger.info("load sequence")
        self.sequence.load(self.region)

    logger.info(f"load data of {len(self.plots)} plots")

    cmds = []
    for p in self.plots:
        assert isinstance(p, PlotInfo), f"unrecognized data type: {type(p)}"
        if self.__n_objs__ / len(self.plots) >= n_jobs > 1:
            if isinstance(p.obj[0].label, list):
                juncs = {}
                for i in p.obj[0].label:
                    juncs.update(self.junctions.get(i, {}))
                p.load(self.region, junctions=juncs, *args, **kwargs)
            else:
                p.load(
                    self.region,
                    n_jobs,
                    junctions=self.junctions.get(p.obj[0].label, {}),
                    *args,
                    **kwargs,
                )
        elif n_jobs > 1:
            temp = copy.deepcopy(kwargs)
            temp["region"] = (
                self.region if p.type != "motif" else self.params[p]["region"]
            )

            if isinstance(p.obj[0].label, list):
                juncs = {}
                for i in p.obj[0].label:
                    juncs.update(self.junctions.get(i, {}))
                temp["junctions"] = juncs
            else:
                temp["junctions"] = self.junctions.get(p.obj[0].label, {})
            cmds.append([p, args, temp])

    if len(cmds) > 0:
        from multiprocessing import Pool

        with Pool(max(1, min(n_jobs, len(self.plots)))) as pool:
            self.plots = pool.map(_tp.__load__, cmds)

    for p in self.plots:
        if n_jobs <= 1:
            if isinstance(p.obj[0].label, list):
                juncs = {}
                for i in p.obj[0].label:
                    juncs.update(self.junctions.get(i, {}))
                p.load(self.region, junctions=juncs, *args, **kwargs)
            else:
                p.load(
                    self.region,
                    junctions=self.junctions.get(p.obj[0].label, {}),
                    *args,
                    **kwargs,
                )

        n_rows, n_height = p.len(annotation_scale, sc_height_ratio=sc_height_ratio)
        plots_n_rows += n_rows
        height_ratio += n_height

        if p.type in ["heatmap", "hic"]:
            plots_n_cols = 2

    prev_len = len(height_ratio)
    plots_n_rows = int(plots_n_rows)
    extra_rows = plots_n_rows - prev_len
    height_ratio += [1 for _ in range(extra_rows)]

    # Layout controls
    xaxis_height_scale = float(kwargs.pop("xaxis_height_scale", 1.0))
    annotation_height_scale = float(kwargs.pop("annotation_height_scale", 1.0))

    if xaxis_height_scale != 1.0 or annotation_height_scale != 1.0:
        anno_rows = (
            self.annotation.len(scale=annotation_scale)
            if self.annotation is not None
            else 0
        )
        stroke_rows = (
            int(max(len(self.stroke) * stroke_scale, 1)) if self.stroke else 0
        )
        link_rows = len(self.link)

        data_rows = prev_len
        xaxis_idx = data_rows + link_rows
        if 0 <= xaxis_idx < len(height_ratio):
            height_ratio[xaxis_idx] *= xaxis_height_scale

        anno_start = xaxis_idx + 1
        for i in range(anno_start, anno_start + anno_rows):
            if 0 <= i < len(height_ratio):
                height_ratio[i] *= annotation_height_scale

    logger.debug(f"plots n_rows={plots_n_rows}; n_cols = {plots_n_cols}")
    logger.info("init graph_coords")
    exon_scale = kwargs.get("exon_scale", 1)
    intron_scale = kwargs.get("intron_scale", 0.5)
    if plots_n_cols > 1 and intron_scale != 1:
        logger.debug("heatmap require intron_scale = 1")
        intron_scale = 1

    self.graph_coords = init_graph_coords(
        self.region, self.exons, exon_scale=exon_scale, intron_scale=intron_scale
    )

    if width and height:
        fig = plt.figure(figsize=[width, height * sum(height_ratio)], dpi=dpi)
    else:
        fig = plt.figure(dpi=dpi)

    if plots_n_cols > 1:
        gs = gridspec.GridSpec(
            plots_n_rows,
            plots_n_cols,
            height_ratios=height_ratio,
            width_ratios=(0.99, 0.01),
            wspace=0.01,
            hspace=grid_hspace,
        )
    else:
        gs = gridspec.GridSpec(
            plots_n_rows,
            plots_n_cols,
            height_ratios=height_ratio,
            wspace=0.7,
            hspace=grid_hspace,
        )

    max_used_y_val, min_used_y_val, same_y_by_groups = {}, {}, {}

    default_y = {}
    if kwargs.get("y_limit"):
        if not os.path.exists(kwargs.get("y_limit")):
            logger.warning(f"{kwargs.get('y_limit')} not exists")
        else:
            logger.info("load y-limit settings from: " + kwargs.get("y_limit"))
            with open(kwargs.get("y_limit")) as r:
                for line in r:
                    if line.startswith("#"):
                        continue
                    line = line.split()
                    if len(line) > 1:
                        try:
                            default_y[line[0]] = [float(x) for x in line[1:]]
                        except Exception as err:
                            logger.warning(
                                f"The y limit of {line[0]} is invalid: {err}"
                            )

    if kwargs.get("same_y") or kwargs.get("same_y_sc"):
        logger.info("--same-y is enabled")
        if kwargs.get("same_y_groups") and os.path.exists(kwargs.get("same_y_groups")):
            logger.info(
                "Reading the same y by groups from: %s"
                % kwargs.get("same_y_groups")
            )
            with open(kwargs.get("same_y_groups")) as r:
                for line in r:
                    if line.startswith("#"):
                        continue
                    line = line.split()
                    same_y_by_groups[line[0]] = (
                        f"samy_y_groups_of_{line[1]} {datetime.now()}"
                    )

        for p in self.plots:
            if p.type in ["density", "site-plot", "line"]:
                for obj in p.obj:
                    if kwargs.get("y_max") is not None:
                        logger.info("")
                    if obj.label in same_y_by_groups:
                        key = same_y_by_groups[obj.label]
                        max_used_y_val[key] = max(
                            max(obj.data.wiggle), max_used_y_val.get(key, 0)
                        )
                        if obj.data.minus is None:
                            min_used_y_val[key] = min(
                                min(obj.data.wiggle), min_used_y_val.get(key, 0)
                            )
                        else:
                            min_used_y_val[key] = min(
                                min(obj.data.minus), min_used_y_val.get(obj.path, 0)
                            )

                    max_used_y_val[obj.path] = max(
                        max(obj.data.wiggle), max_used_y_val.get(obj.path, 0)
                    )
                    if obj.data.minus is None:
                        min_used_y_val[obj.path] = min(
                            min(obj.data.wiggle), min_used_y_val.get(obj.path, 0)
                        )
                    else:
                        min_used_y_val[obj.path] = min(
                            min(obj.data.minus), min_used_y_val.get(obj.path, 0)
                        )

                    if obj.label in default_y:
                        max_used_y_val[obj.path] = default_y[obj.label][0]
                        if len(default_y[obj.label]) > 1:
                            min_used_y_val[obj.path] = default_y[obj.label][1]

                        if same_y_by_groups and obj.label in same_y_by_groups:
                            key = same_y_by_groups[obj.label]
                            max_used_y_val[key] = max_used_y_val[obj.path]
                            min_used_y_val[key] = min_used_y_val[obj.path]

    curr_idx = 0
    for p in self.plots:
        if p.type == "igv":
            ax_var = plt.subplot(gs[curr_idx : curr_idx + p.len(annotation_scale)[0], 0])
        else:
            ax_var = plt.subplot(gs[curr_idx, 0])

        max_y_val_, min_y_val_ = None, None

        if kwargs.get("same_y_sc") and p.obj[0].is_single_cell:
            max_y_val_, min_y_val_ = (
                max_used_y_val[p.obj[0].path],
                min_used_y_val[p.obj[0].path],
            )
        elif kwargs.get("same_y_groups") and p.obj[0].label in same_y_by_groups:
            max_y_val_, min_y_val_ = (
                max_used_y_val[same_y_by_groups[p.obj[0].label]],
                min_used_y_val[same_y_by_groups[p.obj[0].label]],
            )
        elif kwargs.get("same_y"):
            max_y_val_, min_y_val_ = (
                max(max_used_y_val.values()),
                min(min_used_y_val.values()),
            )
        elif default_y:
            if p.type in ["density", "site-plot", "line"]:
                for obj in p.obj:
                    if obj.label in default_y:
                        max_y_val_ = default_y[obj.label][0]
                        if len(default_y[obj.label]) > 1:
                            min_y_val_ = default_y[obj.label][1]
                        if same_y_by_groups and obj.label in same_y_by_groups:
                            key = same_y_by_groups[obj.label]
                            max_y_val_ = max_used_y_val[obj.path]
                            min_y_val_ = min_used_y_val[obj.path]
                        break

        logger.info(
            f"plotting {p.type} at idx: {curr_idx} with height_ratio: {height_ratio[curr_idx]}"
        )

        if p.type == "density":
            if isinstance(p.obj[0], Depth):
                for key, readDepth in p.obj[0].data.items():
                    temp_params = self.params.get(p, {}).copy()
                    if "y_label" not in temp_params:
                        temp_params["y_label"] = key
                    plot_density(
                        ax=ax_var,
                        data=readDepth,
                        region=self.region,
                        graph_coords=self.graph_coords,
                        max_used_y_val=max_y_val_,
                        min_used_y_val=min_y_val_,
                        distance_between_label_axis=distance_between_label_axis,
                        raster=raster,
                        fill_step=fill_step,
                        **temp_params,
                    )
                    curr_idx += 1
                    ax_var = plt.subplot(gs[curr_idx, 0])
                curr_idx -= 1
            else:
                plot_density(
                    ax=ax_var,
                    obj=p.obj[0],
                    graph_coords=self.graph_coords,
                    max_used_y_val=max_y_val_,
                    min_used_y_val=min_y_val_,
                    distance_between_label_axis=distance_between_label_axis,
                    raster=raster,
                    fill_step=fill_step,
                    **self.params.get(p, {}),
                )
        elif p.type == "hic":
            plot_hic(
                ax=ax_var,
                cbar_ax=plt.subplot(gs[curr_idx, 1]),
                obj=p.obj,
                distance_between_label_axis=distance_between_label_axis,
                raster=raster,
                **self.params.get(p, {}),
            )
        elif p.type == "site-plot":
            plot_density(
                ax=ax_var,
                obj=p.obj[0],
                graph_coords=self.graph_coords,
                max_used_y_val=max_y_val_,
                min_used_y_val=min_y_val_,
                distance_between_label_axis=distance_between_label_axis,
                raster=raster,
                **self.params.get(p, {}),
            )
            curr_idx += 1
            plot_site_plot(
                plt.subplot(gs[curr_idx, 0]),
                p.obj[0],
                graph_coords=self.graph_coords,
                raster=raster,
                distance_between_label_axis=distance_between_label_axis,
                **self.params.get(p, {}),
            )
        elif p.type == "heatmap":
            plot_heatmap(
                ax=ax_var,
                cbar_ax=plt.subplot(gs[curr_idx, 1]),
                data=p.data,
                y_label=p.group,
                graph_coords=self.graph_coords,
                raster=raster,
                distance_between_label_axis=distance_between_label_axis,
                **self.params.get(p, {}),
            )
        elif p.type == "line":
            plot_line(
                ax=ax_var,
                data=p.data,
                y_label=p.group,
                max_used_y_val=max_y_val_,
                min_used_y_val=min_y_val_,
                graph_coords=self.graph_coords,
                distance_between_label_axis=distance_between_label_axis,
                **self.params.get(p, {}),
            )
        elif p.type == "igv":
            plot_igv_like(
                ax=ax_var,
                obj=p.data,
                graph_coords=self.graph_coords,
                raster=raster,
                distance_between_label_axis=distance_between_label_axis,
                **self.params.get(p, {}),
            )
        elif p.type == "motif":
            plot_motif(
                ax=ax_var, obj=p.obj[0], graph_coords=self.graph_coords, **self.params[p]
            )
        else:
            raise ValueError(f"unknown plot type {p.type}")

        set_indicator_lines(ax=ax_var, sites=self.sites, graph_coords=self.graph_coords)
        set_focus(ax=ax_var, focus=self.focus, graph_coords=self.graph_coords)

        if p.type != "igv":
            curr_idx += 1
        else:
            curr_idx += p.len(annotation_scale)[0]

    if self.link:
        logger.info(
            f"plotting links at idx: {curr_idx} with height_ratio: {height_ratio[curr_idx]}"
        )
        for link in self.link:
            plot_links(
                ax=plt.subplot(gs[curr_idx : (curr_idx + 1), 0]),
                data=link,
                graph_coords=self.graph_coords,
            )
            curr_idx += 1

    logger.info(
        f"plotting x-axis ticks at idx: {curr_idx} with height_ratio: {height_ratio[curr_idx]}"
    )
    set_x_ticks(
        ax=plt.subplot(gs[curr_idx, 0]),
        region=self.region,
        graph_coords=self.graph_coords,
        sequence=self.sequence.data if self.sequence else None,
        *args,
        **kwargs,
    )
    curr_idx += 1

    if self.annotation is not None:
        logger.info(
            f"plotting annotation at idx: {curr_idx} with height_ratio: {height_ratio[curr_idx]}"
        )
        ax_var = plt.subplot(
            gs[curr_idx : curr_idx + self.annotation.len(scale=annotation_scale), 0]
        )
        plot_annotation(
            ax=ax_var,
            obj=self.annotation,
            graph_coords=self.graph_coords,
            plot_domain=self.annotation.add_domain,
            distance_between_label_axis=distance_between_label_axis,
            **self.params["annotation"],
        )
        ax_var.margins(y=0.0)

        set_indicator_lines(ax=ax_var, sites=self.sites, graph_coords=self.graph_coords)
        set_focus(ax=ax_var, focus=self.focus, graph_coords=self.graph_coords)
        curr_idx += self.annotation.len(scale=annotation_scale)

    if self.stroke:
        logger.info(
            f"plotting stroke at idx: {curr_idx} with height_ratio: {height_ratio[curr_idx]}"
        )
        ax_var = plt.subplot(gs[curr_idx:plots_n_rows, 0])
        plot_stroke(
            ax=ax_var,
            data=self.stroke,
            graph_coords=self.graph_coords,
            *args,
            **kwargs,
        )

    # ---------------------------
    # UPDATED: return fig safely
    # ---------------------------
    if output:
        logger.info(f"saving fig into {output}")
        fig.savefig(output, transparent=False, bbox_inches="tight")
        plt.close(fig)
        logger.info("Plot done")
        return self

    if return_image:
        buf = io.BytesIO()
        if return_image == "png":
            FigureCanvasAgg(fig).print_png(buf)
        elif return_image == "pdf":
            FigureCanvasPdf(fig).print_pdf(buf)
        plt.close(fig)
        logger.info("Plot done")
        return buf

    # Return the live figure so callers can fig.savefig(...) without blanks
    logger.info("Plot done")
    return fig


# Install the monkey-patch
Plot.plot = _compact_plot

## Helper Functions

In [ ]:
#| export
def _versionless(tid: str) -> str:
    """Remove ENS version suffix (if present)"""
    i = tid.rfind(".")
    return tid[:i] if i > 0 and tid[i+1:].isdigit() else tid

In [ ]:
#| export
def _resolve_gene_transcripts(td, gene: str):
    """
    Use TranscriptData to get all transcripts for a gene and return:
        (transcript_ids, gene_id, gene_name)
    `gene` can be either a gene_id or a gene_name.
    """
    tids = td.get_transcripts_by_gene_id(gene)
    gene_id = gene_name = None

    if tids:
        info0 = td.get_transcript_info(tids[0])
        gene_id = info0["gene_id"]
        gene_name = info0["gene_name"]
    else:
        tids = td.get_transcripts_by_gene_name(gene)
        if not tids:
            raise ValueError(f"Gene {gene!r} not found in TranscriptData")
        info0 = td.get_transcript_info(tids[0])
        gene_id = info0["gene_id"]
        gene_name = info0["gene_name"]

    return tids, gene_id, gene_name

In [ ]:
#| export
def _filter_transcripts_for_gene(
    gene_tids: list[str],
    requested: List[str] | None,
) -> List[str] | None:
    """
    Restrict requested transcript IDs to those belonging to the gene.
    Matching is done version-insensitively.
    """
    if requested is None:
        return None

    base_to_full: dict[str, list[str]] = {}
    for tid in gene_tids:
        base = _versionless(tid)
        base_to_full.setdefault(base, []).append(tid)

    found: list[str] = []
    for t in requested:
        if t in gene_tids:
            found.append(t)
        else:
            t_base = _versionless(t)
            found.extend(base_to_full.get(t_base, []))

    if not found:
        warnings.warn(
            "Requested transcripts not found for gene – drawing all isoforms instead."
        )
        return None

    seen = set()
    out: list[str] = []
    for t in found:
        if t not in seen:
            seen.add(t)
            out.append(t)
    return out

In [ ]:
#| export
def _gene_region_from_transcripts(td, gene_tids: list[str], padding: int) -> tuple[str, int, int, str]:
    """Compute gene-wide genomic span from all its transcripts."""
    starts = []
    ends = []
    chrom = None
    strand_char = None

    for tid in gene_tids:
        arr = td.arrays(tid)
        ex = arr["exons"]
        if ex.size == 0:
            continue
        if chrom is None:
            chrom = arr["chrom"]
            strand_char = "+" if arr["strand"] == 1 else "-"
        else:
            if arr["chrom"] != chrom:
                raise ValueError(
                    f"Gene transcripts on different chromosomes: {chrom} vs {arr['chrom']}"
                )
        starts.append(int(ex[:, 0].min()))
        ends.append(int(ex[:, 1].max()))

    if not starts:
        raise ValueError("No exons found for gene transcripts")

    region_start = min(starts) - padding
    region_end   = max(ends) + padding
    return chrom, region_start, region_end, strand_char

In [ ]:
#| export
def _largest_transcript_region(td, tids: list[str]) -> tuple[str, int, int, str, str]:
    """Find the largest transcript and return its genomic region."""
    best_tid = None
    best_span = -1
    best_chrom = None
    best_strand = None
    best_start = None
    best_end = None

    for tid in tids:
        arr = td.arrays(tid)
        ex = arr["exons"]
        if ex.size == 0:
            continue
        start = int(ex[:, 0].min())
        end   = int(ex[:, 1].max())
        span  = end - start
        if span > best_span:
            best_span = span
            best_tid = tid
            best_chrom = arr["chrom"]
            best_strand = "+" if arr["strand"] == 1 else "-"
            best_start = start
            best_end = end

    if best_tid is None:
        raise ValueError("No usable exons to determine largest transcript")

    info = td.get_transcript_info(best_tid)
    label = info.get("transcript_name") or best_tid
    return best_chrom, best_start, best_end, best_strand, label


def _select_top_n_transcripts(
    adata,
    gene_tids: list[str],
    top_n: int,
) -> list[str]:
    """
    Select the top N most abundant transcripts from a gene's isoforms.
    
    Handles version-insensitive matching between gene_tids and adata.var_names.

    Parameters
    ----------
    adata : AnnData
        Annotated data object with transcript counts
    gene_tids : list[str]
        All transcript IDs for the gene (from TranscriptData)
    top_n : int
        Number of top transcripts to select

    Returns
    -------
    list[str]
        Top N transcript IDs by mean expression (in original gene_tids format)
    """
    import numpy as np
    from scipy.sparse import issparse
    
    # Build mapping from versionless ID to full IDs in both gene_tids and adata
    gene_base_to_full = {}
    for tid in gene_tids:
        base = _versionless(tid)
        gene_base_to_full.setdefault(base, []).append(tid)
    
    adata_base_to_full = {}
    for tid in adata.var_names:
        base = _versionless(str(tid))
        adata_base_to_full.setdefault(base, []).append(str(tid))
    
    # Match gene transcripts to adata transcripts (version-insensitive)
    tid_indices = []
    matched_gene_tids = []
    
    for gene_tid in gene_tids:
        gene_base = _versionless(gene_tid)
        
        # Try exact match first
        if gene_tid in adata.var_names:
            tid_indices.append(adata.var_names.get_loc(gene_tid))
            matched_gene_tids.append(gene_tid)
        # Try versionless match
        elif gene_base in adata_base_to_full:
            # Use the first match in adata
            adata_tid = adata_base_to_full[gene_base][0]
            tid_indices.append(adata.var_names.get_loc(adata_tid))
            matched_gene_tids.append(gene_tid)

    if not tid_indices:
        raise ValueError(
            f"None of the gene's {len(gene_tids)} transcripts found in adata.var_names. "
            f"Gene transcripts: {gene_tids[:3]}... "
            f"Example adata transcripts: {list(adata.var_names[:3])}"
        )

    # Calculate mean expression for each matched transcript
    X = adata.X
    if issparse(X):
        means = np.asarray(X[:, tid_indices].mean(axis=0)).ravel()
    else:
        means = X[:, tid_indices].mean(axis=0)
        if hasattr(means, 'ravel'):
            means = means.ravel()

    # Get top_n indices by mean expression
    k = min(top_n, len(means))
    top_indices = np.argsort(means)[::-1][:k]

    # Map back to original gene transcript IDs
    top_tids = [matched_gene_tids[i] for i in top_indices]

    return top_tids

## Main Coverage Plot Function

In [ ]:
#| export
def plot_gene_coverage(
    adata,
    transcript_data,
    gene: str | None = None,
    *,
    chrom: str | None = None,
    start: int | None = None,
    end: int | None = None,
    strand: str | None = None,
    groupby: str | Sequence[str] = ("cell_type",),
    bc_col: str | None = "barcode_raw",
    bam_paths: Mapping[str, str] | None = None,
    bam_column: str = "batch",
    bc_tag: str = "BC",
    padding: int = 1_000,
    figsize: tuple[float, float] = (6, 3),
    font_size: int = 6,
    n_y_ticks: int = 3,
    threshold: int = 1,
    same_y: bool = False,
    raster: bool = True,
    show_site_plot: bool = False,
    transcripts: List[str] | None = None,
    top_n: int | None = None,
    choose_primary: bool = False,
    annotation_scale: float = 0.25,
    intron_scale: float = 0.15,
    exon_width: float = 0.30,
    show_cell_labels: bool = True,
    title: str | None = None,              # kept for API compatibility (unused)
    gtf_file: str | None = None,
    color_key: Optional[str] = "cell_type",
    palette: str | list = "tab10",
    group_color_map: Optional[Dict[str, str]] = None,
    group_order: Optional[List[str]] = None,  # deprecated: use visible_groups
    visible_groups: Optional[List[str]] = None,
    show_title: bool = False,              # kept for API compatibility (ignored)
    distance_between_label_axis: float = 0.2,
    xaxis_height_scale: float = 0.2,
    annotation_height_scale: float = 0.6,
    grid_hspace: float = 0.18,
    verbose: bool = False,
):
    """
    Plot Nanopore read coverage from BAM files with transcript annotations.

    Returns
    -------
    matplotlib.figure.Figure
        The figure object (so the caller can save it via fig.savefig()).
    """
    import logging
    import pandas as pd

    td = transcript_data

    if bam_paths is None:
        raise ValueError("bam_paths (mapping bam_column → bam_path) must be provided")

    if bam_column not in adata.obs.columns:
        raise KeyError(f"{bam_column!r} not found in adata.obs")

    if gtf_file is None:
        raise ValueError("gtf_file must be provided as the GTF annotation path.")

    if transcripts is not None and top_n is not None:
        raise ValueError("Cannot specify both `transcripts` and `top_n`. Choose one.")

    # Normalise groupby: accept a bare string like "cell_type"
    if isinstance(groupby, str):
        groupby = (groupby,)

    # If bc_col is missing from obs, synthesise it from the obs index (cell barcodes)
    _bc_col_synthesised = False
    if bc_col is None or bc_col not in adata.obs.columns:
        bc_col = "_barcode"
        if bc_col not in adata.obs.columns:
            adata = adata.copy()
            adata.obs[bc_col] = adata.obs.index.astype(str)
            _bc_col_synthesised = True

    missing_cols = [c for c in [*groupby, bc_col, bam_column] if c not in adata.obs.columns]
    if missing_cols:
        raise KeyError(f"The following columns are missing from adata.obs: {missing_cols}")

    region_mode = chrom is not None and all(v is not None for v in (start, end, strand))

    transcripts_for_anno: List[str] | None = None
    region_title: str

    if region_mode:
        region_chrom = chrom
        region_start = int(start)
        region_end = int(end)
        region_strand = strand
        region_title = title or f"{chrom}:{start}-{end} ({strand})"

        if transcripts is not None:
            transcripts_for_anno = transcripts[:]
    else:
        if gene is None:
            raise ValueError("Must supply `gene` or (chrom,start,end,strand)")

        gene_tids, gene_id, gene_name = _resolve_gene_transcripts(td, gene)

        if top_n is not None:
            transcripts_for_anno = _select_top_n_transcripts(adata, gene_tids, top_n)
        else:
            transcripts_for_anno = _filter_transcripts_for_gene(gene_tids, transcripts)

        region_chrom, region_start, region_end, region_strand = _gene_region_from_transcripts(
            td, gene_tids, padding
        )
        region_title = title or (gene_name or gene_id or gene)

    # If we picked transcripts, tighten region to the largest selected transcript
    if transcripts_for_anno is not None and len(transcripts_for_anno) > 0:
        region_chrom, region_start, region_end, region_strand, _ = _largest_transcript_region(
            td, transcripts_for_anno
        )

    # Convert transcript IDs -> transcript names for trackplot annotation (if possible)
    if transcripts_for_anno is not None:
        transcripts_for_anno_names: list[str] = []
        for tid in transcripts_for_anno:
            info = td.get_transcript_info(tid)
            name = info.get("transcript_name") or tid
            transcripts_for_anno_names.append(name)
    else:
        transcripts_for_anno_names = None

    obs_cols = list(groupby) + [bam_column, bc_col]
    obs = adata.obs[obs_cols].dropna(subset=[bc_col, bam_column])

    agg_dict = {bc_col: lambda s: set(s)}
    group_cols = list(groupby) + [bam_column]
    groups = obs.groupby(group_cols).agg(agg_dict).reset_index()

    # visible_groups controls both filtering and order (group_order is a backwards-compat alias)
    _visible = visible_groups if visible_groups is not None else group_order
    if _visible is not None:
        first_groupby = list(groupby)[0]
        order_map = {g: i for i, g in enumerate(_visible)}
        groups = groups[groups[first_groupby].isin(_visible)].copy()
        groups["_sort_key"] = groups[first_groupby].map(lambda x: order_map.get(x, len(order_map)))
        groups = groups.sort_values("_sort_key").drop(columns=["_sort_key"]).reset_index(drop=True)

    # Build color map - use group_color_map if provided, otherwise palette
    color_map = {}
    if color_key is not None:
        if color_key not in adata.obs:
            raise KeyError(f"{color_key!r} not found in adata.obs")

        cats = adata.obs[color_key].unique()

        if group_color_map is not None:
            color_map = group_color_map
        else:
            if isinstance(palette, str):
                colors = sns.color_palette(palette, len(cats))
            else:
                colors = list(palette)
            color_map = dict(zip(cats, colors))

    # Suppress trackplot logging unless verbose=True
    trackplot_logger = logging.getLogger("trackplot")
    original_level = trackplot_logger.level

    if not verbose:
        trackplot_logger.setLevel(logging.ERROR)
        print(f"🎨 Plotting coverage for {region_title} ({region_chrom}:{region_start}-{region_end})...")

    fig = None
    try:
        p = Plot(font_family="DejaVu Sans")
        p.set_region(
            chromosome=region_chrom,
            start=region_start,
            end=region_end,
            strand=region_strand,
        )

        for _, row in groups.iterrows():
            bam_id = row[bam_column]
            if bam_id not in bam_paths:
                warnings.warn(f"bam_id {bam_id!r} has no path in bam_paths; skipping")
                continue

            path = bam_paths[bam_id]
            if not os.path.exists(path):
                warnings.warn(f"BAM path {path!r} does not exist; skipping")
                continue

            label = "|".join(str(row[col]) for col in groupby)
            cell_color = color_map.get(row[color_key], "grey") if color_key else "grey"

            p.add_density(
                path=path,
                category="bam",
                label=label,
                barcode=label,
                barcode_groups={label: row[bc_col]},
                barcode_tag=bc_tag,
                font_size=font_size,
                n_y_ticks=n_y_ticks,
                show_y_label=True,
                show_site_plot=show_site_plot,
                color=cell_color,
            )

        p.set_annotation(
            gtf_file,
            transcripts=transcripts_for_anno_names,
            choose_primary=choose_primary,
            show_gene=True,
            show_id=False,
            font_size=font_size,
            exon_width=exon_width,
            color="black",
        )

        maybe_fig = p.plot(
            output=None,
            dpi=300,
            width=figsize[0],
            height=figsize[1],
            raster=raster,
            same_y=same_y,
            threshold=threshold,
            annotation_scale=annotation_scale,
            intron_scale=intron_scale,
            distance_between_label_axis=distance_between_label_axis,
            xaxis_height_scale=xaxis_height_scale,
            annotation_height_scale=annotation_height_scale,
            grid_hspace=grid_hspace,
        )

        # If patched Plot.plot returns a Figure, use it; otherwise fallback to gcf().
        fig = maybe_fig if hasattr(maybe_fig, "savefig") else plt.gcf()

        # Improve x tick label legibility
        for ax in fig.get_axes():
            if ax.get_xlabel() or len(ax.get_xticklabels()) > 0:
                for lab in ax.get_xticklabels():
                    lab.set_rotation(45)
                    lab.set_ha("right")
                    lab.set_fontsize(max(6, font_size - 2))
                xticks = ax.get_xticks()
                if len(xticks) > 6:
                    keep = range(0, len(xticks), 2)
                    ax.set_xticks([xticks[i] for i in keep if i < len(xticks)])

        if not verbose:
            print("✅ Done!")

    finally:
        trackplot_logger.setLevel(original_level)

    return fig

## Example Usage

Load test data and plot coverage for the Myl6 gene.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()